# oxDNA: Melting Temperature Optimization

This notebook optimizes oxDNA1 force-field parameters to match a target melting temperature using DiffTRe. Multiple parallel oxDNA simulations with umbrella sampling are coordinated via Ray.

## What is mythos?

**[Mythos](https://github.com/mythos-bio/mythos)** is a differentiable molecular simulation framework for parameterizing coarse-grained force fields. It provides automatic differentiation through simulation workflows — enabling gradient-based optimization of force field parameters to match experimental or all-atom reference data.

Here we optimize the melting temperature of a short DNA duplex. The melting temperature is extracted from umbrella-sampled simulations at a single temperature, and DiffTRe reweighting allows us to differentiate through the process.

## Imports

In [ ]:
import logging
import typing
from pathlib import Path

import jax
import jax.numpy as jnp
import jax_md
import mythos.energy as jdna_energy
import mythos.energy.dna1 as jdna1_energy
import mythos.optimization.objective as jdna_objective
import mythos.optimization.optimization as jdna_optimization
import optax
import pandas as pd
import ray
from mythos.input import oxdna_input
from mythos.observables.melting_temp import MeltingTemp
from mythos.simulators.oxdna import UmbrellaEnergyInfo, oxDNAUmbrellaSampler
from mythos.ui.loggers.console import ConsoleLogger
from mythos.utils.units import get_kt, get_kt_from_string

jax.config.update("jax_enable_x64", True)
logging.basicConfig(level=logging.INFO)
logging.getLogger("jax").setLevel(logging.WARNING)

## Configuration

In [ ]:
NUM_SIMS = 10
LEARNING_RATE = 1e-3
OPT_STEPS = 100
INPUT_DIR = Path("../../data/templates/tm-6bp-2op")
TARGET_TEMP = get_kt_from_string("31.2C")
KT_RANGE = jnp.linspace(get_kt_from_string("20C"), get_kt_from_string("40C"), NUM_SIMS)
OXDNA_SRC = Path("../../../oxDNA").resolve()

## Initialize Ray

In [ ]:
ray.init(
    log_to_driver=True,
    runtime_env={"env_vars": {"JAX_ENABLE_X64": "True"}},
)

## Load topology and build energy function

In [ ]:
inp = oxdna_input.read_input_dir(INPUT_DIR)

energy_fn = jdna1_energy.create_default_energy_fn(
    topology=inp.topology,
    displacement_fn=jax_md.space.periodic(inp.box_size)[0],
).with_noopt("ss_stack_weights", "ss_hb_weights", "kt"
).with_params(kt=inp.kT)

opt_params = energy_fn.opt_params()

## Create parallel umbrella sampling simulators

We use `oxDNAUmbrellaSampler` — a simulator that handles umbrella sampling state
automatically. Each simulator exposes both a `trajectory` and `energy_info`
observable, and recomputes umbrella weights from the histogram after each run. 
A flattened list of such would then expose:

```py
    [trajectory_1, energy_info_1, trajectory_2, energy_info_2, ..., trajectory_N, energy_info_N],
```

The simulator also returns updated weights in its `SimulatorOutput.state` field,
which will be fed back into the next round of simulations by the optimizer.

In [ ]:
simulators = oxDNAUmbrellaSampler.create_n(
    NUM_SIMS,
    input_dir=INPUT_DIR,
    energy_fn=energy_fn,
    source_path=OXDNA_SRC,
)

## Define the melting temperature objective

DiffTre automatically combines all `SimulatorTrajectory` observables from its
`required_observables` field and feeds them into the first argument of the
`grad_or_loss_function`. They are also available in the full `observables` list
passed to the loss function, which is provided in the same order as
`required_observables`. In order to get the sampled weight level for each
trajectory state, we filter that list by getting all `UmbrellaEnergyInfo`
objects (pandas data frames) and concatenating them, the ordered list means that
the final data frame rows each corresponds to the same state in the concatenated
`trajectory`.


In [ ]:
melting_temp_fn = MeltingTemp(
    rigid_body_transform_fn=lambda x: x,  # unused by MeltingTemp
    sim_temperature=inp.kT,
    temperature_range=KT_RANGE,
    energy_fn=energy_fn,
)


def melting_temp_loss_fn(
    trajectory: jax_md.rigid_body.RigidBody,
    weights: jnp.ndarray,
    _energy_model: jdna_energy.base.ComposedEnergyFunction,
    opt_params,
    observables: list,
) -> tuple[float, tuple[str, typing.Any]]:
    e_info = pd.concat([i for i in observables if isinstance(i, UmbrellaEnergyInfo)])
    obs = melting_temp_fn(
        trajectory, e_info["bond"].to_numpy(), e_info["weight"].to_numpy(), opt_params
    )
    expected_melting_temp = jnp.dot(weights, obs).sum()
    loss = jnp.sqrt((expected_melting_temp - TARGET_TEMP) ** 2)
    return loss, (("melting_temp", expected_melting_temp), {})


required_obs = [name for sim in simulators for name in sim.exposes()]

melting_temp_objective = jdna_objective.DiffTReObjective(
    name="melting_temp",
    required_observables=required_obs,
    logging_observables=["loss", "melting_temp", "neff"],
    grad_or_loss_fn=melting_temp_loss_fn,
    energy_fn=energy_fn,
    min_n_eff_factor=0.95,
    n_equilibration_steps=0,
)

## Run the optimization

We use `RayOptimizer` to coordinate the parallel umbrella sampling simulators.
After each step, we use the callback feature to average and normalize the
umbrella weights across all simulators so that the next round use the same
combined weights, improving sampling consistency. The callback updates the
`component_state` field for all simulators, which the optimizer feeds back into
the next simulator `run` method call.

In [ ]:
opt = jdna_optimization.RayOptimizer(
    objectives=[melting_temp_objective],
    simulators=simulators,
    optimizer=optax.adam(learning_rate=LEARNING_RATE),
    aggregate_grad_fn=lambda grads: grads[0],
    logger=ConsoleLogger(),
)

sim_names = [sim.name for sim in simulators]


def reweight_simulators(component_state):
    """Average umbrella weights across all simulators and write them back."""
    all_weights = pd.concat([component_state[name]["weights"] for name in sim_names])
    mean_weights = all_weights.groupby(all_weights.index.names).mean()
    mean_weights /= mean_weights.query("weights > 0")["weights"].min()
    for name in sim_names:
        component_state[name]["weights"] = mean_weights


def opt_callback(optimizer_output, step):
    """Combine umbrella weights across simulators, then continue."""
    component_state = optimizer_output.state.component_state.copy()
    reweight_simulators(component_state)
    new_state = optimizer_output.state.replace(component_state=component_state)
    return optimizer_output.replace(state=new_state), True  # True to continue optimization


output = opt.run(params=opt_params, n_steps=OPT_STEPS, callback=opt_callback)
print("\nOptimization complete!")
print(f"Final params: {output.opt_params}")